# Archive Note

Historical notebook preserved for provenance. It still references source lighting files that are not included in this repo, so execution is not guaranteed end-to-end.


In [1]:
import pandas as pd
import geopandas as gpd

# Output file: joined lighting points by neighborhood
# 1. Read the CSV with lichtpunten
punten = pd.read_csv("LICHTPUNTEN.csv", sep=";")

# Turn the points into a GeoDataFrame  WGS84 is EPSG 4326
gdf_punten = gpd.GeoDataFrame(
    punten,
    geometry=gpd.points_from_xy(punten["LNG"], punten["LAT"]),
    crs="EPSG:4326"
)

# de gpkg is is verwijderd omdat hij te groot was omdat naar github uploaden te groot was
# 2. Read the wijken from the geopackage
wijken = gpd.read_file("../../data/raw/boundaries/wijken_en_gemeenten.gpkg", layer="wijken")

# Make sure CRS is set correctly  CBS wijkenbuurten is in RD New  EPSG 28992
if wijken.crs is None:
    wijken = wijken.set_crs(epsg=28992)

# Reproject wijken to WGS84 so both layers match
wijken_4326 = wijken.to_crs("EPSG:4326")

# Keep only the fields you care about
wijken_4326 = wijken_4326[["wijkcode", "wijknaam", "gemeentenaam", "geometry"]]

# 3. Spatial join  assign each point to the wijk polygon it falls within
punten_met_wijk = gpd.sjoin(
    gdf_punten,
    wijken_4326,
    how="left",
    predicate="within"
)

# 4. Save back to CSV  without the geometry column
kolommen = [col for col in punten_met_wijk.columns if col != "geometry"]
punten_met_wijk[kolommen].to_csv("../../data/intermediate/LICHTPUNTEN_met_wijk.csv", sep=";", index=False)


FileNotFoundError: [Errno 2] No such file or directory: 'LICHTPUNTEN.csv'